# 03 — Summary Statistics (Person 3)

**Project question:** what drives customer satisfaction (`average_rating`) on Amazon products?

This notebook covers the Submission 1 EDA tasks:
- **4a** summary statistics for the numerical variables
- **4b** analysis of the categorical variables
- **4c** summary tables

All logic lives in `src/stats.py`; this notebook only calls those functions and interprets the results.

**Input:** the cleaned product table from Person 2 (`src/clean.py`), one row per product, 4 source categories.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

from src import stats

## Load the cleaned data

If `data/processed/products_clean.parquet` exists (built from Person 2's `clean_products`), load only the columns this analysis needs — much faster and lighter on memory than loading the text columns. Otherwise rebuild it from the raw file with Person 1's and Person 2's modules.

In [2]:
CLEAN_PATH = "data/processed/products_clean.parquet"
COLS = ["parent_asin", "category", "main_category", "store",
        "average_rating", "rating_number", "log_rating_number",
        "price_num", "log_price", "has_price", "n_features", "n_description"]

if os.path.exists(CLEAN_PATH):
    df = pd.read_parquet(CLEAN_PATH, columns=COLS)
else:
    from src.collect import load_raw
    from src.clean import clean_products
    df = clean_products(load_raw())[COLS]

print("Rows:", f"{len(df):,}", "| Columns:", df.shape[1])
df.head()

Rows: 1,644,621 | Columns: 12


,parent_asin,category,main_category,store,average_rating,rating_number,log_rating_number,price_num,log_price,has_price,n_features,n_description
0,B01CUPMQZE,All_Beauty,All Beauty,Howard Products,4.8,10,2.397895,NaN,NaN,False,0,0
1,B076WQZGPM,All_Beauty,All Beauty,Yes To,4.5,3,1.386294,NaN,NaN,False,0,0
2,B000B658RI,All_Beauty,All Beauty,Levine Health Products,4.4,26,3.295837,NaN,NaN,False,0,0
3,B088FKY3VD,All_Beauty,All Beauty,Cherioll,3.1,102,4.634729,NaN,NaN,False,0,0
4,B07NGFDN6G,All_Beauty,All Beauty,Precision,4.3,7,2.079442,NaN,NaN,False,5,1


In [3]:
df.dtypes

parent_asin                  object
category                     object
main_category        string[python]
store                string[python]
average_rating              float64
rating_number                 int64
log_rating_number           float64
price_num                   float64
log_price                   float64
has_price                      bool
n_features                    int64
n_description                 int64
dtype: object

## 4a — Numerical variables

`numeric_summary` gives count, missing, mean, median, spread, quartiles and **skew** for the three core numeric variables. Skew matters: a large positive skew means a few extreme products pull the mean up, so the median is the better "typical" value.

`price_num` is missing for products with no listed price, so its row describes **priced products only**.

In [4]:
num = stats.numeric_summary(df)
num

,count,n_missing,mean,median,std,min,25%,75%,max,skew
average_rating,1644621.0,0.0,4.222,4.40,0.773,1.00,4.00,4.70,5.00,-1.898
rating_number,1644621.0,0.0,130.095,14.00,849.028,1.00,4.00,54.00,201742.00,47.296
price_num,754211.0,890410.0,53.238,19.49,289.673,0.01,10.99,38.99,129266.64,240.181


The same summary for the log-transformed and text-length columns Person 2 created:

In [5]:
stats.numeric_summary(df, ["log_rating_number", "log_price", "n_features", "n_description"])

,count,n_missing,mean,median,std,min,25%,75%,max,skew
log_rating_number,1644621.0,0.0,2.929,2.708,1.706,0.693,1.609,4.007,12.215,0.748
log_price,754211.0,890410.0,3.181,3.020,1.043,0.010,2.484,3.689,11.770,0.879
n_features,1644621.0,0.0,3.373,5.000,2.251,0.000,1.000,5.000,60.000,-0.367
n_description,1644621.0,0.0,1.834,1.000,4.932,0.000,0.000,1.000,391.000,11.908


## 4b — Categorical variables

Overview first: how many distinct values each categorical column has, the most common value, and how much of it is the `"Unknown"` placeholder Person 2 used for missing values.

In [6]:
stats.categorical_overview(df)

,n_unique,most_common,most_common_pct,n_unknown,pct_unknown
category,4,Toys_and_Games,54.17,0,0.0
main_category,42,Toys & Games,43.48,45532,2.77
store,204586,Unknown,1.56,25688,1.56


### Source category (our 4 categories)

In [7]:
stats.categorical_summary(df, "category")

category: 4 unique values


,count,pct,cum_pct
category,,,
Toys_and_Games,890874,54.17,54.17
Industrial_and_Scientific,427564,26.00,80.17
Musical_Instruments,213593,12.99,93.16
All_Beauty,112590,6.85,100.01


### Amazon's own `main_category` label

In [8]:
stats.categorical_summary(df, "main_category", top=15)

main_category: 42 unique values


,count,pct,cum_pct
main_category,,,
Toys & Games,715061,43.48,43.48
Industrial & Scientific,263258,16.01,59.49
Musical Instruments,185055,11.25,70.74
All Beauty,116598,7.09,77.83
Tools & Home Improvement,88306,5.37,83.2
Amazon Home,72447,4.41,87.61
Unknown,45532,2.77,90.38
AMAZON FASHION,28733,1.75,92.13
Office Products,23951,1.46,93.59


### Store — top 15 by product count
`store` has very many distinct values, so only the top rows are shown.

In [9]:
stats.categorical_summary(df, "store", top=15)

store: 204,586 unique values


,count,pct,cum_pct
store,,,
Unknown,25688,1.56,1.56
Funko,15163,0.92,2.48
uxcell,15065,0.92,3.4
Yu-Gi-Oh!,11295,0.69,4.09
Magic The Gathering,11281,0.69,4.78
LEGO,10958,0.67,5.45
Pokemon,9666,0.59,6.04
Mattel,8284,0.5,6.54
Hasbro,7748,0.47,7.01


## 4c — Summary tables

### Rating by category
Per category: product count, rating centre and spread, `diff_from_overall` (distance from the all-products mean), share of products with a price, price, and median review count.

In [10]:
by_cat = stats.rating_by_category(df)
by_cat

,n_products,mean_rating,diff_from_overall,median_rating,std_rating,pct_has_price,mean_price,median_price,median_rating_number
category,,,,,,,,,
Toys_and_Games,890874,4.275,0.053,4.5,0.701,48.117,35.637,18.95,20.0
Industrial_and_Scientific,427564,4.217,-0.005,4.4,0.847,52.144,60.794,18.27,8.0
Musical_Instruments,213593,4.186,-0.036,4.4,0.797,39.745,127.690,33.99,9.0
All_Beauty,112590,3.883,-0.339,4.0,0.874,15.724,27.256,16.99,9.0


### Products with vs without a listed price
About half the products have no price. Do they rate differently? If yes, anything we say about price only describes the priced half.

In [11]:
stats.rating_by_group(df, "has_price")

,n_products,mean_rating,median_rating,std_rating,pct_has_price,mean_price,median_price,median_rating_number
has_price,,,,,,,,
True,754211,4.338,4.5,0.655,100.0,53.238,19.49,24.0
False,890410,4.123,4.3,0.848,0.0,NaN,NaN,9.0


### Mean rating by price quartile, per category
Quartile cut points are computed over **all priced products together**, so each band covers the same dollar range in every category.

In [12]:
bands = stats.price_band_summary(df)
print("Cut points ($):", bands.attrs["cut_points"])
bands

Cut points ($): [0.01, 10.99, 19.49, 38.99, 129266.64]


category,All_Beauty,Industrial_and_Scientific,Musical_Instruments,Toys_and_Games,All
price_band,,,,,
Q1: $0.01-$10.99,4.055,4.284,4.193,4.373,4.322
Q2: $10.99-$19.49,4.186,4.324,4.251,4.372,4.344
Q3: $19.49-$38.99,4.243,4.347,4.288,4.379,4.359
"Q4: $38.99-$129,266.64",4.243,4.279,4.310,4.367,4.326


Number of products behind each cell above:

In [13]:
bands.attrs["counts"]

category,All_Beauty,Industrial_and_Scientific,Musical_Instruments,Toys_and_Games
price_band,,,,
Q1: $0.01-$10.99,5240,62054,14074,107599
Q2: $10.99-$19.49,4806,54701,14365,114344
Q3: $19.49-$38.99,4804,48624,16697,119405
"Q4: $38.99-$129,266.64",2854,57570,39757,87317


### Mean rating by review volume, per category
This speaks to the project goal of finding **heavily reviewed products that still rate low**. Check the counts table before trusting a small cell.

In [14]:
vol = stats.review_volume_summary(df)
vol

category,All_Beauty,Industrial_and_Scientific,Musical_Instruments,Toys_and_Games,All
volume_band,,,,,
1-9,3.790,4.146,4.131,4.143,4.114
10-99,3.938,4.249,4.192,4.279,4.238
100-999,4.165,4.429,4.399,4.465,4.441
1k-9.9k,4.328,4.518,4.505,4.591,4.567
10k+,4.667,4.534,4.573,4.657,4.622


In [15]:
vol.attrs["counts"]

category,All_Beauty,Industrial_and_Scientific,Musical_Instruments,Toys_and_Games
volume_band,,,,
1-9,56869,227691,110399,312025
10-99,46392,149761,77986,386369
100-999,8643,43857,22345,164429
1k-9.9k,656,5899,2741,27019
10k+,30,356,122,1032


### Top 20 stores by product count
`"Unknown"` is excluded because it is not a real store.

In [16]:
stores = stats.top_stores(df, n=20)
stores

,n_products,main_source_category,mean_rating,median_rating_number,diff_from_overall
store,,,,,
Funko,15163,Toys_and_Games,4.629,115.0,0.407
uxcell,15065,Industrial_and_Scientific,4.166,5.0,-0.056
Yu-Gi-Oh!,11295,Toys_and_Games,4.652,11.0,0.430
Magic The Gathering,11281,Toys_and_Games,4.671,8.0,0.449
LEGO,10958,Toys_and_Games,4.518,82.0,0.297
Pokemon,9666,Toys_and_Games,4.527,12.0,0.305
Mattel,8284,Toys_and_Games,4.344,16.0,0.122
Hasbro,7748,Toys_and_Games,4.261,12.0,0.039
Generic,7025,Toys_and_Games,4.067,5.0,-0.155


Share of all products sold by these 20 stores:

In [17]:
print(f"{100 * stores['n_products'].sum() / len(df):.2f}% of products")

9.23% of products


### Correlations
Spearman (rank-based) by default, because `rating_number` and `price` are heavily skewed. Pearson shown below for comparison. Price correlations use priced products only.

In [18]:
stats.correlation_table(df)

,average_rating,rating_number,log_rating_number,price_num,log_price,n_features,n_description
average_rating,1.000,-0.031,-0.031,0.000,0.000,0.038,0.080
rating_number,-0.031,1.000,1.000,-0.020,-0.020,0.261,-0.039
log_rating_number,-0.031,1.000,1.000,-0.020,-0.020,0.261,-0.039
price_num,0.000,-0.020,-0.020,1.000,1.000,0.038,-0.065
log_price,0.000,-0.020,-0.020,1.000,1.000,0.038,-0.065
n_features,0.038,0.261,0.261,0.038,0.038,1.000,0.296
n_description,0.080,-0.039,-0.039,-0.065,-0.065,0.296,1.000


In [19]:
stats.correlation_table(df, method="pearson")

,average_rating,rating_number,log_rating_number,price_num,log_price,n_features,n_description
average_rating,1.000,0.058,0.172,-0.015,-0.020,0.091,0.019
rating_number,0.058,1.000,0.371,-0.013,-0.015,0.087,-0.006
log_rating_number,0.172,0.371,1.000,-0.053,-0.045,0.259,-0.005
price_num,-0.015,-0.013,-0.053,1.000,0.358,0.004,-0.000
log_price,-0.020,-0.015,-0.045,0.358,1.000,0.033,-0.094
n_features,0.091,0.087,0.259,0.004,0.033,1.000,0.160
n_description,0.019,-0.006,-0.005,-0.000,-0.094,0.160,1.000


**Reading the correlations:** every correlation with `average_rating` is weak (|r| ≤ 0.17). Spearman and Pearson also disagree on review volume (Spearman −0.031 vs Pearson on `log_rating_number` +0.172). Ratings pile up at the top of the scale and many products with only a handful of ratings sit at exactly 5.0, so a single correlation hides the pattern. The review-volume table above is the clearer evidence.

## Preliminary insights

Each insight quotes a number from a table in this notebook; the table is named in brackets.

1. **Ratings are high and bunched at the top.** The mean product rating is **4.22** and the median **4.40**, with a skew of **−1.90**; a quarter of all products sit at 4.7 or above, and three quarters at 4.0 or above. Differences between groups are therefore small in absolute stars, and a 0.3-star gap is large. *[numeric_summary]*

2. **All_Beauty is the clear low-rated category.** It averages **3.883**, **0.339 below** the overall mean, while the other three categories are all within 0.06 of it (Toys_and_Games +0.053, Industrial_and_Scientific −0.005, Musical_Instruments −0.036). Beauty also has the widest spread (std **0.874**). *[rating_by_category]*

3. **More-reviewed products rate higher, in every category.** Mean rating climbs from **4.114** for products with 1–9 ratings to **4.622** for 10,000+. All_Beauty stays the lowest-rated category in every volume band up to 9,999 ratings (e.g. **4.328** at 1k–9.9k vs 4.505–4.591 for the others), which makes it the first candidate for the "heavily reviewed but still low-rated" question. *[review_volume_summary; note only 30 Beauty products have 10k+ ratings]*

4. **Price barely moves ratings, except in Beauty.** Across all priced products the mean rating differs by only **0.037 stars** between the lowest and highest price quartile band (4.322 vs 4.359), and the Spearman correlation is **0.000**. In All_Beauty, the cheapest quartile averages **4.055** vs **4.243** for products above $19.49. *[price_band_summary, correlation_table]*

5. **54% of products have no listed price, and they are not a random half.** Products without a price average **4.123** stars and a median of **9** ratings, vs **4.338** and **24** for priced products. Any price finding describes the better-reviewed, more established half of the catalogue. *[numeric_summary, rating_by_group(has_price)]*

6. **Big stores mostly beat the average; collectible brands lead.** The top 20 stores hold **9.23%** of products, and **16 of 20** rate above the overall mean, led by Magic The Gathering (**4.671**), Yu-Gi-Oh! (**4.652**) and Funko (**4.629**). The four below the mean are generic or bulk sellers: Generic (**4.067**), uxcell (**4.166**), amscan (**4.132**) and Fun Express (**4.148**). *[top_stores]*

**Data notes for the team:** the maximum price is **$129,266.64**, an obvious outlier, which is why medians are reported next to means. Amazon's own `main_category` has **42** values against our 4 source categories (e.g. only 715,061 of 890,874 Toys_and_Games products are labelled "Toys & Games"), so `category` is the reliable grouping. *[numeric_summary, categorical_summary]*

## Next steps (Submission 2)
- Test whether the category gap is statistically meaningful (ANOVA with effect size), not just visible.
- Find individual heavily-reviewed, low-rated products within All_Beauty: the product-level version of insight 3.
- Price–rating correlation per category with confidence intervals, and a decision on how to handle the 54% of products with no price.
- Merge review-level data to add verified-purchase share and review dates.
